<a href="https://colab.research.google.com/github/aleja71291/FDL-EA-20252/blob/main/5_ConvNet_4Class_ADNI1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Intentamos entrenando un modelo compuesto de tres capas convolucionales, seguidas de tres capas densas. Cada capa convolucional tiene un tamaño de Kernel de 3, con padding de 1 (para mantener las dimensiones espaciales). Se aumentaron los canales de forma progresiva entre las capas convolucionales, con el fin de facilitar la extracción jerárquica de características. Se aplicó normalización por lote para promover la convergencia.

In [ ]:
import os
import zipfile
from google.colab import drive

import requests
zip_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/images_Train_1.zip'
csv_url = 'https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/y_resampled_df%20(1).csv'
zip_file_path = 'data.zip'
csv_file_path = 'labels.csv'
extract_path = 'data/'

response = requests.get(zip_url)
with open(zip_file_path, 'wb') as f:
    f.write(response.content)

response = requests.get(csv_url)
with open(csv_file_path, 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import pandas as pd

labels_df = pd.read_csv('labels.csv')
label_mapping = {label: idx for idx, label in enumerate(labels_df['CDGLOBAL'].unique())}
labels_df['label'] = labels_df['CDGLOBAL'].map(label_mapping)
labels_df['filename'] = '_' + labels_df.index.astype(str) + '_image.png'

labels_df.to_csv('labels.csv', index=False)
print("Modified labels.csv has been saved.")

print(labels_df.head())

Modified labels.csv has been saved.
   CDGLOBAL  label      filename
0         1      0  _0_image.png
1         0      1  _1_image.png
2         0      1  _2_image.png
3         0      1  _3_image.png
4         2      2  _4_image.png


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from torchvision import transforms
from PIL import Image
import requests
import zipfile
import shutil

class CustomImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        label = int(self.data_frame.iloc[idx, 0])
        img_name = os.path.join(self.root_dir, f'_{idx}_image.png')
        image = Image.open(img_name).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label

class Net(nn.Module):
    def __init__(self, num_classes=4):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)  # Adjusted dropout

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x


# Transformación de datos
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.15, 0.15), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomResizedCrop(28, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

val_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

full_dataset = CustomImageDataset(csv_file=csv_file_path, root_dir=extract_path, transform=None)
num_classes = len(full_dataset.data_frame.iloc[:, 0].unique())

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) # FIX: Added missing closing parenthesis

# Parámetros de entrenamiento del modelo
num_epochs = 75
best_val_accuracy = 0.0
patience = 10
early_stopping_counter = 0

print(f"Starting training on {device}...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)

    # Loop de validación
    model.eval()
    correct = 0
    total = 0
    val_loss = 0.0
    with torch.no_grad():  # Disable gradient calculation for validation
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_dataset)
    val_accuracy = 100 * correct / total

    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_loss:.4f}, '
          f'Validation Loss: {epoch_val_loss:.4f}, '
          f'Validation Accuracy: {val_accuracy:.2f}%')

    # Early stopping
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        early_stopping_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  --> Saved best model with accuracy: {best_val_accuracy:.2f}%")
    else:
        early_stopping_counter += 1

    if early_stopping_counter >= patience:
        print("Early stopping triggered.")
        break

print("\nTraining complete!")
print(f"Best validation accuracy achieved: {best_val_accuracy:.2f}%")

# Cleanup
os.remove(zip_file_path)
os.remove(csv_file_path)
shutil.rmtree(extract_path)
print("Cleaned up temporary files.")

Starting training on cpu...
Epoch [1/75], Train Loss: 1.4255, Validation Loss: 1.2825, Validation Accuracy: 43.92%
  --> Saved best model with accuracy: 43.92%
Epoch [2/75], Train Loss: 1.2596, Validation Loss: 1.1480, Validation Accuracy: 50.68%
  --> Saved best model with accuracy: 50.68%
Epoch [3/75], Train Loss: 1.1796, Validation Loss: 1.1220, Validation Accuracy: 47.30%
Epoch [4/75], Train Loss: 1.1448, Validation Loss: 1.0451, Validation Accuracy: 62.84%
  --> Saved best model with accuracy: 62.84%
Epoch [5/75], Train Loss: 1.1317, Validation Loss: 1.0287, Validation Accuracy: 64.19%
  --> Saved best model with accuracy: 64.19%
Epoch [6/75], Train Loss: 1.0540, Validation Loss: 1.0006, Validation Accuracy: 60.81%
Epoch [7/75], Train Loss: 1.0174, Validation Loss: 0.8917, Validation Accuracy: 66.22%
  --> Saved best model with accuracy: 66.22%
Epoch [8/75], Train Loss: 0.9780, Validation Loss: 0.9361, Validation Accuracy: 63.51%
Epoch [9/75], Train Loss: 0.9302, Validation Loss: 

Ahora validamos en el set de prueba:

In [ ]:
test_zip_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/images_test_1.zip'
test_csv_url = 'https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/y_test%20(1).csv'
test_zip_file_path = 'test_data.zip'
test_csv_file_path = 'test_labels.csv'
test_extract_path = 'test_data/'

print("Downloading test data...")
response = requests.get(test_zip_url)
with open(test_zip_file_path, 'wb') as f:
    f.write(response.content)

response = requests.get(test_csv_url)
with open(test_csv_file_path, 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile(test_zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(test_extract_path)
print("Test data downloaded and extracted.")

Test data downloaded and extracted.


In [ ]:
import pandas as pd

test_labels_df = pd.read_csv('test_labels.csv')
test_label_mapping = {label: idx for idx, label in enumerate(labels_df['CDGLOBAL'].unique())}
test_labels_df['label'] = test_labels_df['CDGLOBAL'].map(label_mapping)
test_labels_df['filename'] = '_' + test_labels_df.index.astype(str) + '_image.png'

test_labels_df.to_csv('test_labels.csv', index=False)
print("Modified labels.csv has been saved.")

print(test_labels_df.head())

Modified labels.csv has been saved.
   CDGLOBAL  label      filename
0         1      0  _0_image.png
1         1      0  _1_image.png
2         2      2  _2_image.png
3         3      3  _3_image.png
4         1      0  _4_image.png


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score
from PIL import Image
import requests
import zipfile

class Net(nn.Module):
    def __init__(self, num_classes=4):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        x = self.fc3(x)
        return x

class CustomTestImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        label = int(self.data_frame.iloc[idx, 0])
        img_name = os.path.join(self.root_dir, f'_{idx}_image.png')
        image = Image.open(img_name).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label

def evaluate(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    return accuracy, precision, recall


# Transformación del set de prueba (similar al set de entrenamiento)
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

test_dataset = CustomTestImageDataset(csv_file=test_csv_file_path, root_dir=test_extract_path, transform=transform)
test_data_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes_test = len(test_dataset.data_frame.iloc[:, 0].unique())

# Carga del modelo entrenado
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net(num_classes=num_classes_test).to(device)
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)
accuracy, precision, recall = evaluate(model, test_data_loader, device)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

# Clean up
os.remove(test_zip_file_path)
import shutil
shutil.rmtree(test_extract_path)
os.remove(test_csv_file_path)
print("Temporary test files removed.")


Accuracy: 0.4414
Precision: 0.4334
Recall: 0.4414
Temporary test files removed.


A pesar de las transformaciones aplicadas al set de entrenamiento, modificación de la arquitectura del modelo y detención temprana del entrenamiento, la precisión obtenida en el set de prueba, con imágenes nunca antes vistas por el modelo, es mucho menor que la obtenida en el entrenamiento. Lo anterior es un signo de sobreajuste del modelo.